In [6]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from itertools import product

# Load dataset
file_path = "models_results.xlsx"
df = pd.read_excel(file_path)

# Compute Z-scores for all metrics
df['Z_MSE'] = df.groupby('Company')['MSE'].transform(zscore)
df['Z_MAE'] = df.groupby('Company')['MAE'].transform(zscore)
df['Z_MAPE'] = df.groupby('Company')['MAPE'].transform(zscore)
df['Z_R2'] = df.groupby('Company')['R2'].transform(zscore)  # Higher is better

# Generate different weight combinations (sum to 1)
weight_options = np.linspace(0.1, 0.4, 10)  # 10 values from 0.1 to 0.4
weight_combinations = [w for w in product(weight_options, repeat=4) if sum(w) == 1.0]

best_results = None
best_combination = None

# Iterate through different weight combinations
for weights in weight_combinations:
    w_MSE, w_MAE, w_MAPE, w_R2 = weights

    # Compute final score
    df['Final_Score'] = (
        w_MSE * df['Z_MSE'] +
        w_MAE * df['Z_MAE'] +
        w_MAPE * df['Z_MAPE'] -  # Minimize errors
        w_R2 * df['Z_R2']  # Maximize R²
    )

    # Select best model per company, handling NaN in index
    # Drop rows with NaN values in 'Company' and 'Final_Score' before grouping
    valid_df = df.dropna(subset=['Company', 'Final_Score'])
    current_best = valid_df.loc[valid_df.groupby('Company')['Final_Score'].idxmin()]

    # Store the best results based on the most stable weight combination
    if best_results is None or current_best['Final_Score'].mean() < best_results['Final_Score'].mean():
        best_results = current_best.copy()
        best_combination = weights

# Save the best models and weights used
best_results = best_results[['Company', 'Model', 'MSE', 'MAE', 'R2', 'MAPE', 'Final_Score']]
best_results.to_excel("Best_Models_Per_Company_Optimized.xlsx", index=False)

print(f"Best weight combination: MSE={best_combination[0]}, MAE={best_combination[1]}, MAPE={best_combination[2]}, R2={best_combination[3]}")
print("Results saved to 'Best_Models_Per_Company_Optimized.xlsx'")

Best weight combination: MSE=0.4, MAE=0.4, MAPE=0.1, R2=0.1
Results saved to 'Best_Models_Per_Company_Optimized.xlsx'
